In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from torchvision.datasets import EuroSAT
from torchvision.transforms import ToTensor

from euroshap.utils.utils import report_filesize, shape_of_cutout
from euroshap.visualization.utils import plot_cutout

First, let's verify the file size before we download

In [ ]:
url = "http://madm.dfki.de/files/sentinel/EuroSAT.zip"  # URL for the dataset zip file
report_filesize(url)

Let's inspect the labels

In [ ]:
data = EuroSAT(root='../data/processed', transform=ToTensor(), download=True)

labels = data.classes
labels

There are 10 labels. Let's investigate the distribution of these labels in the dataset.

In [ ]:
labels = [label for _, label in data]

label_counts = pd.Series(labels).value_counts()
label_counts

In [ ]:
sns.barplot(data=label_counts)
plt.xlabel('Label')
plt.title('Number of Images with Each Label');

We have a relatively balanced distribution of labels by number. Next let's look at one image per label.

In [ ]:
plot_cutout(data, 0)

In [ ]:
plot_cutout(data, 5_000)

In [ ]:
plot_cutout(data, 7_000)

In [ ]:
plot_cutout(data, 10_000)

In [ ]:
plot_cutout(data, 12_000)

In [ ]:
plot_cutout(data, 15_000)

In [ ]:
plot_cutout(data, 17_000)

In [ ]:
plot_cutout(data, 20_000)

In [ ]:
plot_cutout(data, 22_000)

In [ ]:
plot_cutout(data, 25_000)

In [ ]:
shape_of_cutout(data)

Next let's look at per channel statistics to identify any wildly different intensity ranges between images.

In [ ]:
from torch.utils.data import DataLoader
import numpy as np

def image_stats(dataset, batch_size=1):
    """
    
    """
    loader = DataLoader(dataset=dataset, batch_size=batch_size, shuffle=False)

    
    mean, std = [], []
    
    for img, _ in loader:
        # compute mean and std over H, W: inds 2 and 3
        if batch_size != 1:
            m = img.mean(dim=(0, 2, 3)).numpy()
            s = img.std(dim=(0, 2, 3)).numpy()
            col_shape = np.shape(m)[1]
        else:
            m = img.mean(dim=(2, 3)).numpy()
            s = img.std(dim=(2, 3)).numpy()
            col_shape = np.shape(m)[0]
        mean.append(m)
        std.append(s)
    
    
    means = pd.DataFrame(data=[mean], columns=[f"mean_ch{i}" for i in range(col_shape)])
    stds = pd.DataFrame(data=[std], columns=[f"std_ch{i}" for i in range(col_shape)])

    df = pd.merge(means, stds, left_index=True, right_index=True)
    return df


In [ ]:
# loader = DataLoader(dataset=data, batch_size=1, shuffle=False)
img, _ = data[0]
np.shape(img.mean(dim=(1, 2)).numpy())[0]

In [ ]:
img.mean(dim=(1, 2)).numpy()

In [ ]:
pd.DataFrame([img.mean(dim=(1, 2)).numpy()], columns=range(len(img.mean(dim=(1, 2)).numpy())))

In [ ]:
df = image_stats(data, 256)
df